# Technical Analysis agent (Juliet) – Team 12, QDC Fall 2026

**Question this agent answers:** *which ETFs have the price trend on their side right now, and is it a good moment to enter?*

**Design choices**
- **Pure Python, no AI call.** Same rule as the team graph: *the AI never computes a number*. It also means the signals still work when Gemini is busy or over its free quota.
- **Evidence first.** Each indicator was tested on our 50 ETFs (2022–2026) before being used (sections 4–7). Only the ones with a real, stable predictive power drive the view; the others are just timing notes.
- **Same data and rules as the team:** `data/prices.csv` (Morgane's clean file), today's intraday row never used (orders execute at the T-1 close), `adj_close` for indicators (no fake drop on ex-dividend days), `close` for returns in the backtest (dividends are not credited in the competition).

| Section | What |
|---|---|
| 1 | Setup |
| 2 | Today's signals per ETF |
| 3 | Technical view per theme (same format as the macro agent) |
| 4 | Evidence: which indicators predict the next 5–10 days? |
| 5 | Correlations between indicators: which ones say the same thing? |
| 6 | Correlations between ETFs: move together, opposite, with a delay? (+ divergence strategy test) |
| 7 | Evidence: replaying the competition on history |
| 8 | How to plug the agent into `team_graph.ipynb` |
| 9 | Limits |

## 1. Setup
On Colab: upload `prices.csv` and `technical_agent.py` with the folder icon (or run from the repo, where both are found automatically).

In [1]:
import os, sys, shutil
from pathlib import Path
for folder in [Path.cwd(), *Path.cwd().parents]:           # project folder = the one with data/prices.csv
    if (folder / "data" / "prices.csv").exists():
        os.chdir(folder); break
Path("data").mkdir(exist_ok=True); Path("Technical_code").mkdir(exist_ok=True)
for f, dest in [("prices.csv", "data"), ("technical_agent.py", "Technical_code")]:
    if Path(f).exists() and not (Path(dest) / f).exists():  # Colab: move uploaded files into place
        shutil.move(f, Path(dest) / f)
sys.path.insert(0, "Technical_code")

import pandas as pd
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30); pd.set_option("display.max_colwidth", 80)
from technical_agent import (load_prices, run_technical_agent, ic_table, simulate_competitions,
                             simulation_summary, technical_conflicts, indicator_correlations,
                             combination_test, etf_correlations, lead_lag_test,
                             divergence_test, divergence_summary, METHOD)
prices = load_prices("data/prices.csv")
print(f"{prices['code'].nunique()} ETFs, {prices['date'].min():%Y-%m-%d} to {prices['date'].max():%Y-%m-%d}")

50 ETFs, 2021-01-04 to 2026-11-13


## 2. Today's signals per ETF
- `tech_score`: average percentile rank (0 = worst, 1 = best) of **6-month** and **12-1-month** momentum across our ETFs (12-1 = the last 12 months without the last month, the classic momentum measure).
- `tech_view`: **positive** if `tech_score ≥ 0.67` **and** the price is above its 50-day average; **negative** if `≤ 0.33` **and** below; otherwise **neutral**. Requiring both avoids buying a long-term winner that is already falling (e.g. a top momentum ETF below its 50-day average stays neutral).
- `behaviour`: from each ETF's own history (10-day variance ratio, last 3 years): **trending** ETFs are timed with breakouts, **mean-reverting** ones with pullbacks.
- Leveraged/inverse ETFs (NASDAQ 3X, EURO STOXX 2X, S&P 500 SHORT, STOXX SHORT) are **not ranked**: they follow their unleveraged twin (inverse = flipped view), otherwise the same bet would be counted twice.

In [2]:
result = run_technical_agent(prices)
signals = result["signals"]
cols = ["theme", "tech_view", "confidence", "tech_score", "mom_rank", "mom_6m_%", "mom_12_1_%",
        "vs_ma50_%", "rsi14", "breakout_20d", "behaviour", "timing_note"]
print("As of", result["as_of"], "| views:", signals["tech_view"].value_counts().to_dict())
signals[cols]

As of 2026-11-13 | views: {'neutral': 32, 'negative': 10, 'positive': 8}


,theme,tech_view,confidence,tech_score,mom_rank,mom_6m_%,mom_12_1_%,vs_ma50_%,rsi14,breakout_20d,behaviour,timing_note
code,,,,,,,,,,,,
CYBERSEC,Equity / Tech,neutral,-,0.94,1.0,25.9,56.3,-3.4,43.0,False,trending,-
BIOTECH,Equity / Tech,positive,medium,0.90,2.0,28.8,30.7,3.8,54.0,False,mean-reverting,-
WORLD ENERGY,Energy & commodities,positive,high,0.87,3.0,11.2,51.8,1.3,50.0,False,trending,-
EURO BANKS,Financials,positive,high,0.83,4.0,14.7,29.5,0.4,58.0,False,mixed,-
IBEX 35,European equity,positive,medium,0.76,5.0,13.0,24.5,3.6,64.0,True,mean-reverting,-
ROBOTICS,Equity / Tech,neutral,-,0.76,5.0,4.0,42.5,-5.3,42.0,False,mixed,-
OIL & GAS E&P,Energy & commodities,positive,high,0.73,7.0,4.0,41.5,0.2,49.0,False,trending,-
DIVIDEND LEADERS,Size & style factors,positive,high,0.73,7.0,10.1,24.6,4.2,69.0,True,trending,breakout to a 20-day high (volume not confirmed)
HEALTH CARE,Health care,positive,high,0.71,9.0,28.1,16.8,9.7,71.0,False,mixed,overbought (RSI > 70): risk of buying after a sharp rise


## 3. Technical view per theme
Same format as Gonçalo's macro output (`theme → view, reason, sources`), so the Portfolio Manager and the validation can compare the two views theme by theme.

In [3]:
pd.DataFrame(result["theme_views"]).T[["view", "reason"]]

,view,reason
Defensive sector,neutral,"avg tech_score=0.51, 50% of its ETFs above their 50-day average (STAPLES, DE..."
Digital assets,negative,"avg tech_score=0.18, 0% of its ETFs above their 50-day average (BITCOIN, CRY..."
Emerging markets,neutral,"avg tech_score=0.46, 33% of its ETFs above their 50-day average (BRAZIL, KOR..."
Energy & commodities,positive,"avg tech_score=0.80, 100% of its ETFs above their 50-day average (WORLD ENER..."
Energy transition,negative,"avg tech_score=0.25, 0% of its ETFs above their 50-day average (HYDROGEN, CL..."
Equity / Tech,neutral,"avg tech_score=0.78, 20% of its ETFs above their 50-day average (CYBERSEC, B..."
European equity,neutral,"avg tech_score=0.65, 100% of its ETFs above their 50-day average (IBEX 35, E..."
Financials,positive,"avg tech_score=0.83, 100% of its ETFs above their 50-day average (EURO BANKS)"
Fixed income / Bonds,neutral,"avg tech_score=0.36, 100% of its ETFs above their 50-day average (HIGHY BOND..."
Frontier tech,negative,"avg tech_score=0.33, 0% of its ETFs above their 50-day average (SPACE, QUANTUM)"


## 4. Evidence: which indicators predict the next 5–10 trading days?
**Information coefficient (IC):** every 5 (or 10) days, rank our ETFs by an indicator and by their return over the next 5 (10) days, and measure the rank correlation. A useful indicator has a **mean IC > 0**, a **t-stat above ~2** (not luck) and is positive in most years.

Horizons of 5 and 10 days match the competition: 3–4 days between our trading days, 12 sessions in total. Future returns use **close prices** (what the competition pays).

In [4]:
ic = ic_table(prices)
Path("Technical_code/results").mkdir(parents=True, exist_ok=True)
ic.to_csv("Technical_code/results/ic_table.csv", index=False)
ic

,indicator,horizon_days,mean_IC,t_stat,hit_rate,n_periods,positive_years
5,tech_score,5,0.044,2.64,0.58,248,5/5
0,mom_12_1,5,0.049,2.49,0.58,248,5/5
1,mom_6m,5,0.035,1.98,0.55,248,5/5
4,rsi14,5,0.021,1.32,0.52,248,3/5
3,breakout_20d,5,0.016,1.25,0.52,229,3/5
2,vs_ma50,5,0.021,1.14,0.53,248,3/5
6,low_vol_60d,5,-0.027,-1.08,0.50,211,3/5
12,tech_score,10,0.052,2.05,0.57,124,5/5
7,mom_12_1,10,0.047,1.71,0.53,124,5/5
11,rsi14,10,0.029,1.49,0.56,124,4/5


**Reading:** the combined momentum score (`tech_score`) is the best and most stable indicator (positive in all 5 years, t-stat > 2 at both horizons). Trend (`vs_ma50`), breakout and RSI have a small positive IC but are **not significant**; low volatility is even slightly **negative** in our universe. That is why momentum drives the view and the others are only used as timing notes.

Note: an IC of ~0.05 is small but normal for a real signal in finance: it is a probability edge, not a forecast.

## 5. Correlations between indicators: which ones say the same thing?
Many technical indicators are built from the same prices, so they can be **redundant**: adding two of them means counting the same information twice. Two questions:

- **(a) Do they rank our ETFs the same way?** Average rank correlation between indicators across dates. Close to **+1** = same information; close to **0** = different information; **negative** = opposite.
- **(b) Do they work at the same time?** Correlation of their prediction scores (IC) over time. Low or negative = they succeed at different moments, so they could complement each other.

Extra indicators tested here (not used by the agent): 1- and 3-month momentum, MACD histogram, distance from the 52-week high, 60-day volatility.

In [5]:
corr = indicator_correlations(prices)
corr["values"].to_csv("Technical_code/results/indicator_correlation.csv")
corr["ic"].to_csv("Technical_code/results/indicator_ic_correlation.csv")
print("(a) Do they rank our ETFs the same way? (average rank correlation)")
corr["values"]

(a) Do they rank our ETFs the same way? (average rank correlation)


,tech_score,mom_12_1,mom_6m,mom_3m,mom_1m,vs_ma50,macd_hist,rsi14,dist_52w_high,vol_60d
tech_score,1.00,0.89,0.90,0.56,0.24,0.36,-0.05,0.29,0.49,0.08
mom_12_1,0.89,1.00,0.63,0.38,0.07,0.19,-0.10,0.14,0.38,0.08
mom_6m,0.90,0.63,1.00,0.64,0.37,0.48,0.02,0.40,0.51,0.06
mom_3m,0.56,0.38,0.64,1.00,0.49,0.68,-0.00,0.55,0.49,0.03
mom_1m,0.24,0.07,0.37,0.49,1.00,0.84,0.46,0.76,0.40,0.02
vs_ma50,0.36,0.19,0.48,0.68,0.84,1.00,0.33,0.82,0.49,0.00
macd_hist,-0.05,-0.10,0.02,-0.00,0.46,0.33,1.00,0.51,0.13,0.03
rsi14,0.29,0.14,0.40,0.55,0.76,0.82,0.51,1.00,0.53,-0.07
dist_52w_high,0.49,0.38,0.51,0.49,0.40,0.49,0.13,0.53,1.00,-0.58
vol_60d,0.08,0.08,0.06,0.03,0.02,0.00,0.03,-0.07,-0.58,1.00


In [6]:
print("(b) Do they predict well at the same time? (correlation of their 10-day IC over time)")
corr["ic"]

(b) Do they predict well at the same time? (correlation of their 10-day IC over time)


,tech_score,mom_12_1,mom_6m,mom_3m,mom_1m,vs_ma50,macd_hist,breakout_20d,rsi14,dist_52w_high,vol_60d
tech_score,1.00,0.91,0.92,0.53,0.12,0.21,-0.17,-0.03,0.16,0.33,0.11
mom_12_1,0.91,1.00,0.69,0.33,-0.08,-0.01,-0.23,-0.17,-0.02,0.19,0.16
mom_6m,0.92,0.69,1.00,0.64,0.31,0.39,-0.05,0.11,0.33,0.41,0.05
mom_3m,0.53,0.33,0.64,1.00,0.51,0.70,-0.07,0.23,0.54,0.50,-0.11
mom_1m,0.12,-0.08,0.31,0.51,1.00,0.87,0.50,0.45,0.76,0.36,-0.03
vs_ma50,0.21,-0.01,0.39,0.70,0.87,1.00,0.34,0.47,0.83,0.49,-0.13
macd_hist,-0.17,-0.23,-0.05,-0.07,0.50,0.34,1.00,0.41,0.52,0.08,0.00
breakout_20d,-0.03,-0.17,0.11,0.23,0.45,0.47,0.41,1.00,0.62,0.35,-0.15
rsi14,0.16,-0.02,0.33,0.54,0.76,0.83,0.52,0.62,1.00,0.54,-0.22
dist_52w_high,0.33,0.19,0.41,0.50,0.36,0.49,0.08,0.35,0.54,1.00,-0.77


**Reading**
- **Short-term trend indicators are almost the same thing:** 1-month momentum, price vs 50-day average and RSI correlate **0.76–0.84**. Using two of them (e.g. MA *and* RSI) only counts the same information twice. MACD is related to them too (≈ 0.5).
- **Long-term momentum (12-1) is a different signal:** almost uncorrelated with the short-term group (0.07 with 1-month momentum, 0.19 with MA50, −0.10 with MACD).
- **Negative correlation:** volatile ETFs tend to be far below their 52-week high (−0.58).
- **(b)** When 12-1 momentum predicts well, MACD and breakouts tend to do slightly *worse* (−0.23, −0.17): they work at different times. Could combining them help? Test below.

In [7]:
combo = combination_test(prices)
combo.to_csv("Technical_code/results/combination_test.csv", index=False)
combo

,signal,horizon_days,mean_IC,t_stat,positive_years
0,12-1 momentum only,5,0.051,2.99,5/5
1,"tech_score (12-1 + 6m, used by the agent)",5,0.045,2.67,5/5
2,12-1 + vs_ma50,5,0.045,2.69,5/5
3,12-1 + rsi14,5,0.043,2.71,5/5
4,12-1 + mom_1m,5,0.043,2.63,5/5
5,12-1 + macd_hist,5,0.027,1.54,4/5
6,12-1 + vol_60d,5,0.050,2.32,5/5
7,12-1 momentum only,10,0.055,2.22,5/5
8,"tech_score (12-1 + 6m, used by the agent)",10,0.052,2.06,5/5
9,12-1 + vs_ma50,10,0.048,1.99,4/5


**Reading:** no combination clearly beats **12-1 momentum alone**. Adding a short-term indicator gives about the same IC and makes 2023 worse; adding MACD clearly hurts. The agent's `tech_score` (12-1 + 6-month) is close to 12-1 alone.

**Conclusion for our model:** more indicators ≠ better. Keep **one long-term momentum signal** as the main driver, use **one** short-term indicator (MA50) only as a filter, and treat RSI / breakout as timing notes, never add them to the score.

## 6. Correlations between ETFs: together, opposite, with a delay?
Three questions:
1. **Which ETFs move together (positive) or opposite (negative)?** Daily returns, last 12 months.
2. **Is there a delay?** Does ETF A today predict ETF B 1–2 days later (lead-lag)?
3. **Divergence strategy (Juliet's idea):** two ETFs usually move together (or opposite). A has clearly moved over the last 5 days, but B has **not** moved as much as it usually does (e.g. A rose, B "should have fallen" but did not). Does B **catch up** afterwards? If yes, we could sell (or buy) B before it moves.

In [8]:
ec = etf_correlations(prices)
ec["matrix"].to_csv("Technical_code/results/etf_correlation_1y.csv")
print("Share of pairs with correlation < -0.3:", ec["share_negative_below_-0.3"],
      "| excluding leveraged/inverse ETFs:", ec["share_negative_excl_leveraged"])
print(); print("Most POSITIVE pairs (last 12 months):"); display(ec["most_positive"])
print("Most NEGATIVE pairs (last 12 months):"); display(ec["most_negative"])

Share of pairs with correlation < -0.3: 0.046 | excluding leveraged/inverse ETFs: 0.018

Most POSITIVE pairs (last 12 months):


,A,B,correlation,mechanical
801,EURO STOXX 2X,EURO STOXX 50,1.00,True
1401,NASDAQ,NASDAQ 3X,0.97,True
1809,S&P 500,WORLD,0.97,False
802,EURO STOXX 2X,EUROPE 600,0.96,True
851,EURO STOXX 50,EUROPE 600,0.96,False
1068,GOLD MINERS,SILVER MINERS,0.96,False
1565,OIL & GAS E&P,WORLD ENERGY,0.93,False
760,EURO BANKS,IBEX 35,0.92,False
1402,NASDAQ,NASDAQ INCOME,0.91,False
1408,NASDAQ,S&P 500,0.91,False


Most NEGATIVE pairs (last 12 months):


,A,B,correlation,mechanical
826,EURO STOXX 2X,STOXX SHORT,-1.00,True
875,EURO STOXX 50,STOXX SHORT,-1.00,True
924,EUROPE 600,STOXX SHORT,-0.96,True
1267,IBEX 35,STOXX SHORT,-0.89,True
777,EURO BANKS,STOXX SHORT,-0.87,True
1458,NASDAQ 3X,S&P 500 SHORT,-0.80,True
1858,S&P 500 SHORT,WORLD,-0.77,True
2106,STOXX SHORT,WORLD SMALL CAP,-0.76,True
2103,STOXX SHORT,WORLD,-0.75,True
821,EURO STOXX 2X,S&P 500 SHORT,-0.74,True


**Reading:** many ETFs move together (WORLD–S&P 500 0.97, gold miners–silver miners 0.96, energy–oil & gas 0.93): owning both is almost the same bet (useful for the Risk agent). **Real opposite movements are rare:** all strongly negative pairs involve the two **inverse** ETFs (STOXX SHORT, S&P 500 SHORT), which are built to move opposite. Between normal ETFs, almost no pair has a correlation below −0.3.

In [9]:
ll = lead_lag_test(prices)
ll.to_csv("Technical_code/results/lead_lag_test.csv", index=False)
print("Strongest 'A today -> B 1-2 days later' pairs found in 2022-2024, and the same relation in 2025-2026:")
print("average |correlation| before:", round(ll["corr_before"].abs().mean(), 3), "| after:", round(ll["corr_after"].abs().mean(), 3))
ll.head(15)

Strongest 'A today -> B 1-2 days later' pairs found in 2022-2024, and the same relation in 2025-2026:
average |correlation| before: 0.259 | after: 0.048


,lag_days,leader,follower,corr_before,corr_after,days_of_data_before
0,1,NASDAQ INCOME,KOREA,-0.385,-0.048,39
1,1,NASDAQ INCOME,CLEAN ENERGY,-0.372,-0.020,39
2,1,NASDAQ INCOME,INDIA,0.355,-0.045,39
3,1,NASDAQ INCOME,COPPER MINERS,-0.347,0.034,39
4,1,NASDAQ INCOME,COPPER LVG,-0.340,0.077,39
5,1,OIL & GAS E&P,NASDAQ INCOME,0.308,0.049,39
6,1,NASDAQ INCOME,S&P 500 SHORT,0.281,0.025,39
7,1,STOXX SHORT,NASDAQ INCOME,0.264,0.077,39
8,1,EURO STOXX 50,NASDAQ INCOME,-0.260,-0.071,39
9,1,EURO STOXX 2X,NASDAQ INCOME,-0.255,-0.078,39


**Reading:** the "delays" that look strongest in 2022–2024 almost **disappear** in 2025–2026 (average |correlation| falls from ≈ 0.26 to ≈ 0.05). Most of them involve NASDAQ INCOME, which had only about 2 months of data before 2025: with so few days, big correlations appear by chance. Lesson: a pattern found by searching among ~2,300 pairs must be **checked on later data** before trusting it.

In [10]:
events = divergence_test(prices)
summary = divergence_summary(events)
summary.to_csv("Technical_code/results/divergence_test.csv", index=False)
print(len(events), "events (A moved > 1 standard deviation in 5 days, |correlation| >= 0.5)")
summary

38192 events (A moved > 1 standard deviation in 5 days, |correlation| >= 0.5)


,pair_type,horizon,events,catch_up_hit_rate,base_rate_B_down,IC_gap
0,negative pair,B_next_5d,878,0.494,0.507,0.012
1,negative pair,B_next_10d,878,0.494,0.511,0.023
2,positive pair,B_next_5d,5787,0.487,0.420,-0.015
3,positive pair,B_next_10d,5787,0.501,0.409,-0.010
4,same index (leveraged/inverse twin),B_next_5d,214,0.486,0.446,0.068
5,same index (leveraged/inverse twin),B_next_10d,214,0.514,0.434,0.094


In [11]:
# Juliet's exact case: NEGATIVE pair, A went up, B did NOT fall (gap_z >= 1.5) -> 'sell B before it falls'
neg = events[events["pair_type"] == "negative pair"]
case = neg[(neg["A_move_z"] >= 1) & (neg["gap_z"] >= 1.5)]
print(f"{len(case)} cases. B fell in the next 5 days: {(case['B_next_5d'] < 0).mean():.1%} "
      f"(vs {(neg['B_next_5d'] < 0).mean():.1%} for any negative-pair event)")
print("Share of these cases where B is an inverse ETF:", f"{case['B'].isin(['STOXX SHORT', 'S&P 500 SHORT']).mean():.0%}")
case.groupby(["A", "B"]).size().sort_values(ascending=False).head(8)

279 cases. B fell in the next 5 days: 50.2% (vs 50.7% for any negative-pair event)
Share of these cases where B is an inverse ETF: 53%


A                 B            
NASDAQ 3X         STOXX SHORT      7
NASDAQ            STOXX SHORT      7
ROBOTICS          S&P 500 SHORT    6
                  STOXX SHORT      6
S&P 500           STOXX SHORT      6
KOREA             S&P 500 SHORT    5
NASDAQ 3X         S&P 500 SHORT    5
DIVIDEND LEADERS  STOXX SHORT      5
dtype: int64

**Reading (honest):** the catch-up bet is right only **about 50% of the time**, for positive pairs, negative pairs and even same-index twins: no better than a coin flip. In Juliet's exact case (A up, B "should have fallen" but did not), B fell **50% of the time**, the same as without any signal. Most of these cases involve the inverse ETFs, whose drift comes from the general market trend, not from a delayed reaction.

**Why it does not work here:** our ETFs are liquid and arbitraged: prices of linked ETFs adjust on the **same day**, so there is no reliable delay to exploit at a 5–10-day horizon. (Such strategies can work intraday or in less liquid markets.)

**What we keep:** the correlation matrix itself is useful for **risk**: highly correlated ETFs (> 0.9) should count as the same bet when we diversify.

## 7. Evidence: replaying the competition on history
We replay **our competition** every week since 2022 (243 windows), with its real rules: 5 trading days in 12 sessions (same pattern as 7–21 Dec), max 5 orders per day, orders at the previous close, valuation at close prices (no dividends), no leverage.

Pure-technical test strategy: equal weight in the 5 ETFs with the best `tech_score` that are above their 50-day average; on later trading days, replace at most 2 that dropped out. Benchmark: 100% WORLD.

In [12]:
sim = simulate_competitions(prices)
sim.to_csv("Technical_code/results/competition_replay.csv", index=False)
print(simulation_summary(sim))
sim.assign(year=pd.to_datetime(sim["start"]).dt.year).groupby("year")[["strategy_%", "world_%", "excess_%"]].mean().round(2)

{'windows': 243, 'beat_WORLD_%': 49.4, 'avg_excess_%': 0.6, 'median_excess_%': -0.07, 'avg_strategy_%': 1.22, 'avg_world_%': 0.61, 'worst_strategy_%': -12.32, 'worst_world_%': -12.9, 'best_strategy_%': 19.52}


,strategy_%,world_%,excess_%
year,,,
2022,0.37,-0.31,0.68
2023,-0.07,0.85,-0.92
2024,1.57,1.25,0.32
2025,2.93,0.37,2.56
2026,1.25,0.85,0.40


**Reading (honest):** on average the technical strategy earns more than WORLD per competition window, but it beats WORLD only about **half of the time** (median excess ≈ 0): the gain comes from a few strong trending periods (2025), and it lost in 2023. Alone, momentum is **not** enough to beat the benchmark reliably over 2–3 weeks.

**Consequence for our system:** the technical signal is one input for the Portfolio Manager, next to the macro view and the risk rules, not an autopilot. It is most useful to (1) avoid ETFs in a clear downtrend (negative views) and (2) choose between ETFs inside a theme the team already likes.

## 8. How to plug it into `team_graph.ipynb` (Francesco)
Francesco's `build_pm_prompt(pf, macro, technical_signals=...)` already has a slot for this agent. Three small changes:

In [13]:
import json
# --- 1) a node, after macro (uses the same in-memory `prices` as the other agents) ---------------------------
#   State: add the key   technical: dict
def technical_node(state):
    print("1b) Technical: momentum, trend, breakout, RSI (Python, no AI)...", flush=True)
    r = run_technical_agent(prices)
    return {"technical": {"as_of": r["as_of"], "summary": r["summary"], "theme_views": r["theme_views"],
                          "signals": json.loads(r["signals"].reset_index().to_json(orient="records", date_format="iso"))}}

#   graph:  g.add_node("technical", technical_node)
#           g.add_edge("macro", "technical");  g.add_edge("technical", "portfolio_manager")   (instead of macro -> portfolio_manager)

# --- 2) in portfolio_manager_node: pass the summary to the prompt ----------------------------------------------
#   r = ask_portfolio_manager(state["portfolio"], state["macro"],
#                             technical_signals=state["technical"]["summary"], feedback=feedback)

# --- 3) in validate_node: same idea as the macro check ---------------------------------------------------------
#   buying an ETF with a NEGATIVE technical view needs "Contrary to technical signal because ..." in its reason
#   sig = pd.DataFrame(state["technical"]["signals"]).set_index("code")
#   errors += technical_conflicts(state["weights"], sig, state["reasons"], current_weights(state["portfolio"]))

# quick test of the check (no AI): buying RARE EARTHS-like negative ETFs without a reason is flagged
neg = signals.index[signals["tech_view"] == "negative"][:1].tolist()
print("Test - buy a negative-view ETF without explanation:", technical_conflicts({neg[0]: 0.10}, signals, {}))
print("Test - same, with 'Contrary to technical signal because ...':",
      technical_conflicts({neg[0]: 0.10}, signals, {neg[0]: "Contrary to technical signal because the macro view is strong"}))
print("\nWhat the Portfolio Manager receives (first lines):\n" + result["summary"][:1200] + " ...")

Test - buy a negative-view ETF without explanation: ["GOLD: its technical view is NEGATIVE (tech_score=0.31 (momentum rank 34 of 45), mom_6m=-9.6%, mom_12_1=5.6%, vs_ma50=-4.3%, rsi14=41) - remove it or explain in its reason: 'Contrary to technical signal because ...'"]
Test - same, with 'Contrary to technical signal because ...': []

What the Portfolio Manager receives (first lines):
Computed in Python by the Technical Analysis agent from adjusted closing prices (T-1 close). tech_score = average percentile rank (0 worst, 1 best) of 6-month and 12-1-month momentum across our ETFs: in our 2022-2026 backtest this was the ONLY indicator with a stable predictive power for the next 5-10 trading days. tech_view: positive = tech_score >= 0.67 AND price above its 50-day average; negative = tech_score <= 0.33 AND below; else neutral. Leveraged/inverse ETFs follow their unleveraged twin (inverse = flipped). breakout_20d, rsi14 and timing_note are weak signals on their own: use them only to time 

## 9. Limits (for the report)
- **Momentum fails at turning points** (e.g. sharp reversals after news): the macro view and the human approval are there for that.
- **Short histories** (QUANTUM, BITCOIN, NASDAQ INCOME…) have a 6-month signal only; ETFs with < 127 days of data get no view.
- **Leveraged ETFs** reset daily: their long-run return is not 3× the index when markets go sideways, so they never get "high" confidence.
- The behaviour type (trending / mean-reverting) is estimated from 3 years of data and can change.
- **Multiple testing:** we tested 7 indicators × 2 horizons; we only rely on the one that is significant at both horizons and positive every year, to reduce the risk of a lucky result.